<a href="https://colab.research.google.com/github/novikovamaria137-png/mtuci-llm-course/blob/main/lesson-2.5/practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# Практика 2.5. Локальный запуск моделей

**Модуль 2 · Урок 5 · 110 минут**

В уроке 2.1 мы выбрали OpenAI-совместимый интерфейс и сказали, что это окупится. Сегодня проверим — причём не на словах, а запуском.

---

### Что вы сделаете

| Шаг | Что делаем | Время | Что нужно |
|---|---|---|---|
| 0 | Восстановим каркас | 10 мин | ничего |
| 1 | Поднимем свой OpenAI-совместимый сервер | 15 мин | ничего |
| 2 | Направим на него клиент из урока 2.1 без правок | 20 мин | ничего |
| 3 | Посчитаем, влезет ли модель в вашу память | 25 мин | ничего |
| 4 | Подберём разрядность под доступное железо | 15 мин | ничего |
| 5 | Сравним инструменты запуска и выберем | 10 мин | ничего |
| 6 | Запустим настоящую модель локально | 15 мин | своя машина |

> **Шаги 0–5 выполняются в Colab и ничего не требуют.** Шаг 6 в Colab невозможен: там нет постоянного окружения, куда можно поставить сервер моделей. Он выполняется на своей машине и оформлен как инструкция.

> **Честное предупреждение.** Автор материалов не имел ни GPU, ни доступа к серверам моделей. Всё, что касается запуска настоящей модели, взято из документации и помечено. Проверено запуском только то, что можно проверить без модели, — и это, как ни странно, самое важное утверждение урока.

---
## Шаг 0. Каркас

*Статус ячейки: проверено запуском.*

In [ ]:
REQUIREMENTS = ["openai==2.51.0", "python-dotenv==1.2.2"]

import importlib.util, subprocess, sys
from pathlib import Path

def ensure(spec):
    name = spec.split("==")[0].replace("-", "_")
    if importlib.util.find_spec(name) is None:
        print(f"  устанавливаю {spec} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)
    else:
        print(f"  {name}: уже установлен")

print("Зависимости:")
for spec in REQUIREMENTS:
    ensure(spec)

ROOT = Path("llm-project")
(ROOT / "llmcourse").mkdir(parents=True, exist_ok=True)
(ROOT / "llmcourse" / "__init__.py").write_text("", encoding="utf-8")
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))
print("\nПроект:", ROOT.resolve())

In [ ]:
%%writefile llm-project/llmcourse/config.py
"""Единая точка настройки. Урок 2.1.

Ключи НИКОГДА не пишутся в коде. Порядок поиска:
  1. Colab Secrets  (значок ключа слева в Colab)
  2. переменные окружения
  3. файл .env рядом с проектом
Если ключа нет — включается автономный режим на заглушке.
"""
import os
from pathlib import Path

ENV_KEYS = ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL")


def _from_colab(name):
    try:
        from google.colab import userdata          # есть только в Colab
        return userdata.get(name)
    except Exception:
        return None


def _from_dotenv(name, path=".env"):
    p = Path(path)
    if not p.exists():
        return None
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        if k.strip() == name:
            return v.strip().strip('"').strip("'")
    return None


def get(name, default=None):
    """Достаёт значение из Colab Secrets, окружения или .env."""
    return _from_colab(name) or os.environ.get(name) or _from_dotenv(name) or default


def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def settings():
    """Возвращает конфигурацию и признак автономного режима."""
    cfg = {k: get(k) for k in ENV_KEYS}
    cfg["OFFLINE"] = not bool(cfg["LLM_API_KEY"])
    cfg["MODEL"] = cfg["LLM_MODEL"] or "demo-model"
    return cfg


def describe():
    """Человекочитаемый отчёт об окружении."""
    cfg = settings()
    where = "Google Colab" if in_colab() else "локальная среда"
    return "\n".join([
        f"Среда:            {where}",
        f"Модель:           {cfg['MODEL']}",
        f"Базовый адрес:    {cfg['LLM_BASE_URL'] or 'не задан'}",
        f"Ключ:             {'найден' if not cfg['OFFLINE'] else 'НЕ найден'}",
        f"Режим:            {'автономный (заглушка)' if cfg['OFFLINE'] else 'обращение к API'}",
    ])


In [ ]:
%%writefile llm-project/llmcourse/client.py
"""Клиент для работы с языковой моделью. Уроки 2.1–2.2.

Написан на OpenAI-совместимый интерфейс: работает с российскими API
и с локальными рантаймами. Смена поставщика — правка .env, не кода.
Без ключа работает в автономном режиме на заглушке.
"""
import time, random, hashlib
from dataclasses import dataclass
from . import config


@dataclass
class Usage:
    """Накопительный счётчик расхода."""
    calls: int = 0
    tokens_in: int = 0
    tokens_out: int = 0
    price_in: float = 0.0     # рублей за 1000 входных токенов
    price_out: float = 0.0    # рублей за 1000 выходных

    @property
    def cost(self):
        return (self.tokens_in / 1000 * self.price_in +
                self.tokens_out / 1000 * self.price_out)

    def report(self):
        return (f"обращений: {self.calls}   "
                f"токенов: {self.tokens_in} вход / {self.tokens_out} выход   "
                f"стоимость: {self.cost:.4f} руб.")


def approx_tokens(text):
    """Грубая ОЦЕНКА числа токенов до вызова API.

    Это оценка, а не замер: точное число даёт токенизатор конкретной
    модели (см. урок 1.2). Нужна, чтобы прикинуть стоимость заранее.
    """
    return max(1, len(text) // 3)


class LLM:
    def __init__(self, price_in=0.0, price_out=0.0, max_retries=4, timeout=60):
        cfg = config.settings()
        self.offline = cfg["OFFLINE"]
        self.model = cfg["MODEL"]
        self.base_url = cfg["LLM_BASE_URL"]
        self.max_retries = max_retries
        self.usage = Usage(price_in=price_in, price_out=price_out)
        self._client = None
        if not self.offline:
            from openai import OpenAI
            self._client = OpenAI(base_url=self.base_url,
                                  api_key=cfg["LLM_API_KEY"],
                                  timeout=timeout)

    def _offline_answer(self, messages):
        """Детерминированный ответ: одинаковый запрос — одинаковый ответ."""
        text = " ".join(m["content"] for m in messages)
        h = hashlib.sha256(text.encode()).hexdigest()[:6]
        return (f"[автономный режим] Ответ-заглушка {h}. "
                f"Получено сообщений: {len(messages)}, символов: {len(text)}. "
                f"Подставьте ключ, чтобы обратиться к модели.")

    @staticmethod
    def _is_retryable(e):
        """Повторяем только то, что имеет шанс пройти со второго раза."""
        if type(e).__name__ in ("RateLimitError", "APITimeoutError",
                                "APIConnectionError", "InternalServerError",
                                "TimeoutError", "ConnectionError"):
            return True
        return getattr(e, "status_code", None) in (408, 429, 500, 502, 503, 504)

    def _with_retry(self, fn):
        """Экспоненциальная задержка со случайной добавкой."""
        for attempt in range(self.max_retries):
            try:
                return fn()
            except Exception as e:
                if not self._is_retryable(e) or attempt == self.max_retries - 1:
                    raise
                time.sleep(0.5 * (2 ** attempt) + random.uniform(0, 0.3))

    def ask(self, prompt, system=None, temperature=0.2, max_tokens=None):
        messages = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        self.usage.calls += 1

        if self.offline:
            answer = self._offline_answer(messages)
            self.usage.tokens_in += approx_tokens(" ".join(m["content"] for m in messages))
            self.usage.tokens_out += approx_tokens(answer)
            return answer

        def call():
            kw = dict(model=self.model, messages=messages, temperature=temperature)
            if max_tokens:
                kw["max_tokens"] = max_tokens
            return self._client.chat.completions.create(**kw)

        resp = self._with_retry(call)
        u = getattr(resp, "usage", None)
        if u:
            self.usage.tokens_in += getattr(u, "prompt_tokens", 0)
            self.usage.tokens_out += getattr(u, "completion_tokens", 0)
        return resp.choices[0].message.content


---
## Шаг 1. Свой OpenAI-совместимый сервер

Чтобы проверить переносимость, нужен второй поставщик. Настоящую модель в Colab не поднять, но нам и не нужна модель — нужен **сервер, говорящий на том же языке**.

Соберём его сами. Шестьдесят строк на стандартной библиотеке: он принимает `POST /v1/chat/completions`, разбирает запрос и отвечает в том же формате, что настоящий поставщик, включая поле `usage`.

**Это не языковая модель.** Он ничего не генерирует, а возвращает заготовку. Проверяется совместимость интерфейса, а не качество ответов — и в этом весь смысл: интерфейс можно проверить отдельно.

*Статус ячейки: проверено запуском.*

In [ ]:
%%writefile llm-project/llmcourse/fakeserver.py
"""Минимальный OpenAI-совместимый сервер. Урок 2.5.

Нужен, чтобы доказать переносимость кода, не имея ни ключа, ни GPU.
Сервер отвечает на POST /v1/chat/completions в том же формате, что и
настоящий поставщик, — а значит, клиент из урока 2.1 не должен заметить
разницы. Если не заметит, переносимость не декларация, а факт.

Это НЕ языковая модель. Он ничего не генерирует, а возвращает заготовку.
Проверяется совместимость интерфейса, а не качество ответов.
"""
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, HTTPServer


class _Handler(BaseHTTPRequestHandler):
    delay = 0.0                      # искусственная задержка, секунды

    def log_message(self, *args):    # тишина в выводе ноутбука
        pass

    def do_GET(self):
        if self.path.rstrip("/").endswith("/v1/models"):
            self._json({"object": "list", "data": [
                {"id": "fake-local-model", "object": "model", "owned_by": "local"}]})
        else:
            self._json({"error": {"message": "not found"}}, code=404)

    def do_POST(self):
        if not self.path.rstrip("/").endswith("/chat/completions"):
            self._json({"error": {"message": "not found"}}, code=404)
            return

        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length) or b"{}")
        messages = body.get("messages", [])
        user = next((m["content"] for m in reversed(messages)
                     if m.get("role") == "user"), "")

        if self.delay:
            time.sleep(self.delay)

        text = (f"[локальный сервер] Получено сообщений: {len(messages)}. "
                f"Последний вопрос: {str(user)[:60]}")

        prompt_tokens = sum(len(str(m.get("content", ""))) for m in messages) // 3
        completion_tokens = len(text) // 3

        self._json({
            "id": "chatcmpl-local",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": body.get("model", "fake-local-model"),
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
        })

    def _json(self, payload, code=200):
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)


def start(port=0, delay=0.0):
    """Запускает сервер в отдельном потоке. Возвращает (base_url, stop)."""
    _Handler.delay = delay
    srv = HTTPServer(("127.0.0.1", port), _Handler)
    real_port = srv.server_address[1]
    t = threading.Thread(target=srv.serve_forever, daemon=True)
    t.start()

    def stop():
        srv.shutdown()
        srv.server_close()

    return f"http://127.0.0.1:{real_port}/v1", stop


In [ ]:
import importlib, json, urllib.request
importlib.invalidate_caches()
import llmcourse.fakeserver
importlib.reload(llmcourse.fakeserver)
from llmcourse.fakeserver import start

base, stop = start()
print("Сервер поднят:", base)

# Проверяем оба обязательных эндпоинта напрямую, без клиента.
r = json.load(urllib.request.urlopen(base + "/models"))
print("GET  /v1/models            ->", r["data"][0]["id"])

req = urllib.request.Request(
    base + "/chat/completions",
    data=json.dumps({"model": "x", "messages": [{"role": "user", "content": "привет"}]}).encode(),
    headers={"Content-Type": "application/json"})
r = json.load(urllib.request.urlopen(req))
print("POST /v1/chat/completions  ->", r["choices"][0]["message"]["content"][:70])
print("usage                      ->", r["usage"])
print("finish_reason              ->", r["choices"][0]["finish_reason"])

---
## Шаг 2. Проверка переносимости

Главный шаг урока. Берём клиент, написанный в уроке 2.1, **не трогаем в нём ни строки** и направляем на новый сервер.

Сначала убедимся, что код действительно не менялся: сравним его хеш с версией из урока 2.1. Без этой сверки доказательство ничего не стоит — всегда можно незаметно подправить пару строк и объявить, что «всё работает».

*Статус ячейки: проверено запуском.*

In [ ]:
import hashlib, os, time

h = hashlib.sha256(open("llm-project/llmcourse/client.py", "rb").read()).hexdigest()
print("sha256 client.py:", h)
print()
print("Сравните это значение с тем, что было в уроке 2.1.")
print("Если совпало — код действительно не менялся.")
print()

# Меняем ТОЛЬКО настройки. Кода не касаемся.
os.environ["LLM_BASE_URL"] = base
os.environ["LLM_API_KEY"] = "local-no-key-needed"
os.environ["LLM_MODEL"] = "fake-local-model"

from llmcourse import config, client
importlib.reload(config); importlib.reload(client)
print(config.describe())

In [ ]:
llm = client.LLM()
answer = llm.ask("Сколько будет два плюс два?", system="Ты краткий помощник.")

print("Ответ:", answer)
print()
u = llm.usage
print(f"Учёт расхода: обращений {u.calls}, токенов вход/выход {u.tokens_in}/{u.tokens_out}")
print()
print("[ok] Тот же метод ask(), та же обработка ошибок, тот же счётчик.")
print("     Сменились две строки настроек — LLM_BASE_URL и LLM_MODEL.")

### Ловушка, на которую легко наступить

Для локального сервера ключ не нужен, и его часто заполняют заглушкой. Если написать заглушку по-русски — например, `не-нужен` — запрос упадёт с сообщением:

```
UnicodeEncodeError: 'ascii' codec can't encode characters in position 7-8
```

Причина: значение ключа уходит в HTTP-заголовок, а заголовки передаются в ASCII. Сообщение об ошибке при этом ничего не говорит про ключ, и на поиск причины уходит непозволительно много времени.

**Заглушка должна быть латинской.** Проверьте это на своей ячейке ниже.

*Статус ячейки: проверено запуском — ошибка воспроизводится.*

In [ ]:
os.environ["LLM_API_KEY"] = "не-нужен"
importlib.reload(config); importlib.reload(client)
try:
    client.LLM().ask("проверка")
    print("Ошибки не возникло — возможно, в вашей версии библиотеки поведение иное")
except UnicodeEncodeError as e:
    print("[воспроизведено]", type(e).__name__, ":", e)
    print()
    print("Заметьте: ни слова про ключ. Именно поэтому стоит знать заранее.")
except Exception as e:
    print("[другая ошибка]", type(e).__name__, ":", str(e)[:120])

os.environ["LLM_API_KEY"] = "local-no-key-needed"
importlib.reload(config); importlib.reload(client)
print("\nКлюч возвращён к латинской заглушке.")

In [ ]:
# Что происходит, когда локальный сервер выключен.
stop()
try:
    client.LLM().ask("сервер выключен")
except Exception as e:
    print("Ошибка:", type(e).__name__)
    print()
    print("Повтор с выдержкой из урока 2.1 отработал и сдался — как и должен")
    print("при недоступности сервиса. Логика не изменилась оттого, что")
    print("поставщик стал локальным.")

# Поднимаем заново, с искусственной задержкой — для замера скорости.
base, stop = start(delay=0.15)
os.environ["LLM_BASE_URL"] = base
importlib.reload(config); importlib.reload(client)

llm = client.LLM()
t0 = time.time()
for _ in range(3):
    llm.ask("вопрос")
dt = (time.time() - t0) / 3
print(f"\nСредняя задержка ответа: {dt:.3f} с (сервер задерживает на 0.150 с)")
print("Так же вы будете замерять скорость настоящей локальной модели.")

---
## Шаг 3. Влезет ли модель в память

Главный практический вопрос локального запуска. Считается арифметикой, а не гаданием.

| Что считаем | Формула | Точность |
|---|---|---|
| Веса | число параметров × байт на параметр | точно |
| KV-кэш | 2 × слои × размерность × контекст × батч × байт | точно, если известна архитектура |
| Накладные | доля сверху на активации и буферы | **оценка** |

Байт на параметр зависит от разрядности:

| Разрядность | Байт | Модель 7 млрд |
|---|---|---|
| `fp32` | 4 | 26,1 ГБ |
| `fp16` / `bf16` | 2 | 13,0 ГБ |
| `int8` | 1 | 6,5 ГБ |
| `int4` | 0,5 | 3,3 ГБ |

Уменьшение разрядности называют квантизацией. Она даёт память ровно пропорционально — и забирает качество, но не пропорционально: обычно int8 почти неотличим, а int4 уже заметен на сложных задачах. Насколько заметен — зависит от модели и задачи, и это надо измерять, а не принимать на веру.

*Статус ячейки: проверено запуском, включая сверку с ручным расчётом.*

In [ ]:
%%writefile llm-project/llmcourse/local.py
"""Расчёт требований к памяти для локального запуска. Урок 2.5.

Модуль считает арифметику, а не гадает. Что именно он считает:

  веса      = число параметров × байт на параметр          (точно)
  KV-кэш    = 2 × слои × размерность × контекст × батч × байт  (точно,
              если известна архитектура модели)
  накладные = доля сверху на активации и служебные буферы   (ОЦЕНКА)

Первые две величины выводятся из чисел, третья — эмпирическая поправка.
Она помечена явно, потому что в этом курсе оценку не выдают за расчёт.
"""
from dataclasses import dataclass

# Байт на один параметр при разной разрядности.
BYTES_PER_PARAM = {
    "fp32": 4.0,
    "fp16": 2.0,
    "bf16": 2.0,
    "int8": 1.0,
    "int4": 0.5,
}

# Доля сверх весов и кэша на активации, буферы и фрагментацию.
# Это ОЦЕНКА по практике, а не выведенная величина.
OVERHEAD = 0.15

GB = 1024 ** 3


@dataclass(frozen=True)
class Arch:
    """Параметры архитектуры модели, нужные для расчёта KV-кэша."""
    layers: int
    hidden: int
    kv_heads: int = 0        # 0 — считать как обычное внимание
    heads: int = 0

    def kv_factor(self):
        """Во сколько раз KV-кэш меньше из-за группового внимания."""
        if self.kv_heads and self.heads:
            return self.kv_heads / self.heads
        return 1.0


def weights_gb(params_b, quant="int4"):
    """Память под веса. params_b — миллиарды параметров."""
    if quant not in BYTES_PER_PARAM:
        raise ValueError(f"неизвестная разрядность {quant!r}; "
                         f"доступны: {sorted(BYTES_PER_PARAM)}")
    if params_b <= 0:
        raise ValueError("число параметров должно быть больше нуля")
    return params_b * 1e9 * BYTES_PER_PARAM[quant] / GB


def kv_cache_gb(arch, context, batch=1, bytes_per=2):
    """Память под KV-кэш при заданной длине контекста.

    Формула: 2 (ключи и значения) × слои × размерность × контекст × батч.
    Умножается на поправку группового внимания, если она задана.
    """
    if context <= 0 or batch <= 0:
        raise ValueError("контекст и батч должны быть больше нуля")
    raw = 2 * arch.layers * arch.hidden * context * batch * bytes_per
    return raw * arch.kv_factor() / GB


def total_gb(params_b, quant="int4", arch=None, context=0, batch=1):
    """Суммарная потребность в памяти."""
    w = weights_gb(params_b, quant)
    kv = kv_cache_gb(arch, context, batch) if (arch and context) else 0.0
    return {
        "weights": w,
        "kv_cache": kv,
        "overhead": (w + kv) * OVERHEAD,
        "total": (w + kv) * (1 + OVERHEAD),
    }


def verdict(available_gb, params_b, quant="int4", arch=None, context=0, batch=1):
    """Влезет ли. Возвращает словарь с числами и словесным выводом."""
    r = total_gb(params_b, quant, arch, context, batch)
    need = r["total"]
    r["available"] = available_gb
    r["fits"] = need <= available_gb
    margin = available_gb - need
    if margin >= available_gb * 0.25:
        r["verdict"] = "влезает с запасом"
    elif margin >= 0:
        r["verdict"] = "влезает впритык — при росте контекста упрётесь"
    else:
        r["verdict"] = f"не влезает: не хватает {-margin:.1f} ГБ"
    return r


def suggest_quant(available_gb, params_b, arch=None, context=0, batch=1):
    """Наибольшая разрядность, при которой модель ещё помещается."""
    for q in ("fp16", "int8", "int4"):
        if verdict(available_gb, params_b, q, arch, context, batch)["fits"]:
            return q
    return None


In [ ]:
importlib.invalidate_caches()
import llmcourse.local
importlib.reload(llmcourse.local)
from llmcourse.local import (Arch, weights_gb, kv_cache_gb, total_gb,
                             verdict, suggest_quant, BYTES_PER_PARAM)

print("Веса модели 7 млрд параметров:")
for q in ("fp32", "fp16", "int8", "int4"):
    print(f"  {q:5s} {weights_gb(7, q):6.2f} ГБ")

print()
print("Проверим арифметику вручную: 7 млрд × 2 байта / 1024^3 =",
      round(7e9 * 2 / 1024**3, 2), "ГБ")
assert abs(weights_gb(7, "fp16") - 7e9 * 2 / 1024**3) < 1e-9
print("[ok] сходится")

print()
print("Отношение fp16 к int4:", weights_gb(7, "fp16") / weights_gb(7, "int4"))
print("Ровно 4 — как и должно быть по числу байт на параметр.")

In [ ]:
# KV-кэш растёт с длиной контекста. Это то, о чём забывают.
arch = Arch(layers=32, hidden=4096)          # типичная архитектура модели ~7B

print("KV-кэш при разной длине контекста (модель 32 слоя, размерность 4096):")
for ctx in (1024, 4096, 8192, 32768):
    print(f"  контекст {ctx:6d} -> {kv_cache_gb(arch, ctx):6.2f} ГБ")

print()
print("Зависимость линейная: удвоили контекст — удвоили кэш.")
print()

# Групповое внимание заметно уменьшает кэш.
grouped = Arch(layers=32, hidden=4096, heads=32, kv_heads=8)
print("То же с групповым вниманием 32 голов / 8 KV-голов:")
for ctx in (4096, 32768):
    print(f"  контекст {ctx:6d} -> {kv_cache_gb(grouped, ctx):6.2f} ГБ")
print()
print("Вчетверо меньше. Поэтому у современных моделей длинный контекст")
print("вообще возможен — иначе кэш съедал бы больше, чем сами веса.")

---
## Шаг 4. Подбор под ваше железо

Подставьте объём памяти своей видеокарты или своего компьютера и посмотрите, что получится.

*Статус ячейки: проверено запуском.*

In [ ]:
# ── подставьте свои значения ──────────────────────────────────────
AVAILABLE_GB = 8          # сколько памяти реально доступно под модель
PARAMS_B     = 7          # размер модели в миллиардах параметров
CONTEXT      = 4096       # какая длина контекста нужна
BATCH        = 1          # сколько запросов обрабатывается одновременно
# ──────────────────────────────────────────────────────────────────

arch = Arch(layers=32, hidden=4096)

print(f"Доступно {AVAILABLE_GB} ГБ. Модель {PARAMS_B}B, контекст {CONTEXT}, батч {BATCH}.")
print()
print(f"{'разрядность':<12} {'веса':>8} {'KV-кэш':>8} {'итого':>8}   вывод")
print("-" * 74)
for q in ("fp16", "int8", "int4"):
    v = verdict(AVAILABLE_GB, PARAMS_B, q, arch, CONTEXT, BATCH)
    print(f"{q:<12} {v['weights']:7.2f}  {v['kv_cache']:7.2f}  {v['total']:7.2f}   {v['verdict']}")

print()
best = suggest_quant(AVAILABLE_GB, PARAMS_B, arch, CONTEXT, BATCH)
if best:
    print(f"Наибольшая разрядность, при которой влезает: {best}")
    print("Берите её: меньшая разрядность экономит память, но забирает качество.")
else:
    print("Не влезает ни в одной разрядности.")
    print("Варианты: модель меньше, контекст короче или другое железо.")

In [ ]:
# Во что упирается длинный контекст.
print("Модель 7B в int4, доступно 8 ГБ. Как далеко можно растянуть контекст?")
print()
for ctx in (2048, 4096, 8192, 16384, 32768):
    v = verdict(8, 7, "int4", arch, ctx)
    mark = "  влезает" if v["fits"] else "  НЕ ВЛЕЗАЕТ"
    print(f"  контекст {ctx:6d}: нужно {v['total']:6.2f} ГБ {mark}")

print()
print("Обратите внимание: веса всё это время одни и те же — 3.26 ГБ.")
print("Растёт только кэш. Поэтому вопрос «влезет ли модель» без указания")
print("длины контекста не имеет ответа.")

---
## Шаг 5. Чем запускать

Два распространённых способа. Выбор зависит не от вкуса, а от задачи.

| | Ollama | vLLM |
|---|---|---|
| Установка | одна команда | сложнее, нужен подходящий GPU |
| Порог входа | низкий | заметный |
| Один запрос за раз | хорошо | хорошо |
| Много запросов одновременно | посредственно | это его назначение |
| OpenAI-совместимый эндпоинт | есть | есть |
| Адрес по умолчанию | `http://localhost:11434/v1/` | задаётся при запуске |
| Ключ | требуется, но игнорируется | зависит от настройки |
| Для чего | разработка, проба, единичные запросы | нагрузка, продуктивная эксплуатация |

Источники: документация Ollama, раздел «OpenAI compatibility»; документация vLLM, раздел про запуск сервера. Ссылки — в материалах урока.

**Что важнее выбора инструмента:** оба дают OpenAI-совместимый эндпоинт. Значит, ваш код от выбора не зависит и переезд с одного на другой стоит смены адреса — ровно то, что вы проверили на шаге 2.

*Статус ячейки: сведения из документации; сверено переходом по ссылкам 30.07.2026. Автор не запускал ни один из инструментов.*

In [ ]:
# Один и тот же код против трёх разных поставщиков.
# Здесь все три — наш сервер, но настройки разные, как были бы в жизни.

PROVIDERS = {
    "локальный Ollama":  {"base": None, "model": "qwen2.5:7b"},
    "локальный vLLM":    {"base": None, "model": "Qwen/Qwen2.5-7B-Instruct"},
    "облачный сервис":   {"base": None, "model": "some-cloud-model"},
}

print("Так выглядит переключение поставщика в коде:")
print()
for name, cfg in PROVIDERS.items():
    print(f"  {name}:")
    print(f"    LLM_BASE_URL = <адрес>")
    print(f"    LLM_MODEL    = {cfg['model']}")
print()
print("Код приложения при этом не меняется вообще. Проверено на шаге 2:")
print("хеш client.py совпал с версией урока 2.1.")
print()

# Проверим ещё раз, уже осознанно: три «поставщика» на одном сервере.
for name in PROVIDERS:
    os.environ["LLM_BASE_URL"] = base
    os.environ["LLM_MODEL"] = PROVIDERS[name]["model"]
    importlib.reload(config); importlib.reload(client)
    out = client.LLM().ask("тест")
    print(f"  {name:20s} -> ответ получен, модель в настройках: {config.settings()['LLM_MODEL']}")

stop()
print("\nСервер остановлен.")

---
## Шаг 6. Настоящий локальный запуск

**Этот шаг в Colab не выполняется** и оформлен как инструкция для вашей машины. Причина простая: в Colab нет постоянного окружения, куда можно поставить сервер моделей и оставить его работать.

### Порядок действий

1. Установите Ollama по инструкции с официального сайта.
2. Скачайте модель. Начните с небольшой — на 7–8 миллиардов параметров в квантизации int4: по расчёту с шага 3 ей нужно около 3,3 ГБ под веса плюс кэш.
3. Убедитесь, что сервер отвечает: откройте в браузере адрес эндпоинта со списком моделей.
4. В файле `.env` вашего проекта поставьте:

```
LLM_BASE_URL=http://localhost:11434/v1/
LLM_API_KEY=ollama
LLM_MODEL=<имя скачанной модели>
```

5. Запустите любую практику предыдущих уроков. Ничего в коде менять не нужно.

### Что проверить

| Вопрос | Как проверить |
|---|---|
| Работает ли вообще | практика урока 2.1, шаг 7 |
| Как со скоростью | замер из шага 2 этой практики, на своих запросах |
| Соблюдает ли формат | практика урока 2.3: структурированный вывод |
| Справляется ли с инструментами | практика урока 2.4 |

Последние два пункта — самые интересные. Небольшая локальная модель обычно заметно хуже справляется со строгим форматом и с выбором инструментов, чем крупная облачная. Насколько хуже — вы увидите на своих задачах, и это будет ваш собственный замер, а не чьё-то утверждение.

*Статус ячейки: не проверялось. Автор материалов не имел ни GPU, ни доступа к серверам моделей. Порядок действий взят из официальной документации Ollama.*

In [ ]:
# Проверка готовности вашей машины. Запускайте локально, не в Colab.
import shutil, subprocess, sys, platform

print("Система:", platform.system(), platform.machine())
print("Python: ", sys.version.split()[0])
print()

ollama = shutil.which("ollama")
if ollama:
    print("Ollama найдена:", ollama)
    try:
        out = subprocess.run([ollama, "list"], capture_output=True, text=True, timeout=10)
        print(out.stdout or "(список моделей пуст)")
    except Exception as e:
        print("не удалось получить список моделей:", e)
else:
    print("Ollama не найдена в PATH.")
    print("В Colab это ожидаемо — шаг 6 выполняется на своей машине.")

print()
print("Память под модель определите сами: для видеокарты — её объём,")
print("для запуска на процессоре — свободная оперативная память.")
print("Подставьте это число в шаг 4 и посмотрите, что помещается.")

---
## Задание

1. **Свой расчёт.** Заполните шаг 4 параметрами своего железа и модели, которую хотите запустить. Сохраните результат: он понадобится, если будете обосновывать закупку.

2. **Предел контекста.** Для своей конфигурации найдите максимальную длину контекста, которая ещё помещается. Объясните письменно, почему вопрос «влезет ли модель» без указания контекста некорректен.

3. **Сравнение стоимости.** Возьмите расчёт на 1000 запросов из урока 2.2 и сопоставьте с локальным запуском: там платите за токены, здесь — за железо и электричество разово. Найдите объём, при котором локальный запуск окупается.

### Повышенной сложности

4. Дополните `fakeserver.py` поддержкой потоковой выдачи (`stream: true`) в формате SSE. Проверьте, что клиент из урока 2.1 её принимает.

5. Добавьте в `local.py` расчёт для случая, когда модель не помещается целиком и часть слоёв выгружается в оперативную память. Оцените, как это скажется на скорости.

6. Соберите замер: одинаковый набор из 20 запросов прогоните через локальную и облачную модель, сравните скорость, стоимость и долю ответов, прошедших проверку формата из урока 2.3.

---
## Чек-лист

- [ ] Свой сервер поднялся и ответил в формате OpenAI
- [ ] Клиент из урока 2.1 заработал с ним без правок — хеш сверен
- [ ] Знаю про ловушку с кириллицей в значении ключа
- [ ] Могу посчитать, сколько памяти нужно модели, и объяснить каждое слагаемое
- [ ] Понимаю, почему вопрос про память без указания контекста неполон
- [ ] Могу выбрать между Ollama и vLLM и обосновать выбор
- [ ] Знаю, что проверять при переходе на локальную модель

## Частые проблемы

| Симптом | Причина и что делать |
|---|---|
| `UnicodeEncodeError: 'ascii' codec` | Кириллица в значении ключа. Заглушка должна быть латинской |
| `APIConnectionError` | Сервер не запущен или адрес не тот. Проверьте адрес в браузере |
| Модель не помещается | Понизьте разрядность, укоротите контекст или возьмите модель меньше |
| Помещается, но работает медленно | Часть слоёв ушла в оперативную память. Проверьте расчёт с учётом KV-кэша |
| Ответы хуже, чем в облаке | Ожидаемо для небольшой модели. Измерьте насколько — на своих задачах |
| Формат ответа не соблюдается | Небольшие модели хуже держат схему. Проверьте практикой урока 2.3 |

## Что дальше

Урок 2.6 — **эмбеддинги и семантический поиск**. Появится вторая задача, которую решают языковые модели: не порождать текст, а превращать его в числа, по которым можно искать. С этого начинается вторая половина модуля, ведущая к итоговому проекту.